# Importar librerias

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, CLIPVisionModel, AutoModel
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Motor encendido usando: {device}")

# Semillas
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import torch
import gc

# Forzamos al recolector de basura de Python a limpiar variables sueltas
gc.collect()

# Vaciamos la memoria caché de la tarjeta gráfica
torch.cuda.empty_cache()

print(" Memoria de la tarjeta gráfica limpiada.")

# Cargar Dataset

tarea 1

In [ ]:
print("Cargando datos y generando Soft Labels...")

# Cargo el JSON
ruta_json = 'EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json'
with open(ruta_json, 'r', encoding='utf-8') as f:
    datos_crudos = json.load(f)

# Convierto a DF de pandas
df_completo = pd.DataFrame.from_dict(datos_crudos, orient='index')

# Función para calcular soft labels
def calcular_soft_label(votos):
    # Si fallase lo pongo en 0.5/0.5
    if not isinstance(votos, list) or len(votos) == 0:
        return [0.5, 0.5]

    total = len(votos)
    # Cuento cuantos yes y no hay
    yes_count = sum(1 for v in votos if v == 'YES')
    no_count = total - yes_count

    # Calculo y devuelvo la probabilidad de cada uno
    return [no_count / total, yes_count / total]

# Aplicola función.
df_completo['soft_label'] = df_completo['labels_task2_1'].apply(calcular_soft_label)

print("Soft Labels calculadas. Ejemplo del primer meme:")
print(f"Votos: {df_completo['labels_task2_1'].iloc[0]}")
print(f"Soft Label [NO, YES]: {df_completo['soft_label'].iloc[0]}")

tarea 2

In [ ]:
import json
import pandas as pd
import numpy as np

print("Cargando datos y preparando Tarea 2 (Direct vs Judgemental)...")

# Cargo el JSON usando tu ruta original
ruta_json = 'EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json'
with open(ruta_json, 'r', encoding='utf-8') as f:
    data_json = json.load(f)

# Defino las clases de la Tarea 2
clases_task2 = ['DIRECT', 'JUDGEMENTAL']
lista_datos_t2 = []

# Itero y filtro
for meme_id, info in data_json.items():
    votos_t1 = info.get('labels_task2_1', [])
    votos_yes = votos_t1.count('YES')
    votos_no = votos_t1.count('NO')
    
    # FILTRO: Solo continuamos si la mayoría dijo YES
    if votos_yes > votos_no:
        votos_t2 = info.get('labels_task2_2', [])
        
        # LIMPIEZA: Ignoro los guiones "-" de quienes votaron NO
        votos_t2_validos = [voto for voto in votos_t2 if voto in clases_task2]
        total_validos = len(votos_t2_validos)
        
        if total_validos > 0:
            counts = {clase: votos_t2_validos.count(clase) for clase in clases_task2}
            
            # Soft Labels [Prob_DIRECT, Prob_JUDGEMENTAL]
            soft_label_t2 = [counts[clase] / total_validos for clase in clases_task2]
            
            lista_datos_t2.append({
                'id_EXIST': info['id_EXIST'],
                'text': info['text'],
                'path_memes': info['path_memes'],
                'labels_task2_2': votos_t2,
                'soft_label': soft_label_t2,
                'sensorial': info.get('sensorial', {}) 
            })

# Convierto a DF de pandas 
df_completo = pd.DataFrame(lista_datos_t2)

print(f"Total de memes originales en el JSON: {len(data_json)}")
print(f"Memes SÍ Sexistas listos para la Tarea 2: {len(df_completo)}")

print("\nSoft Labels calculadas para la Tarea 2. Ejemplo del primer meme filtrado:")
print(f"Soft Label [DIRECT, JUDGEMENTAL]: {df_completo['soft_label'].iloc[0]}")

tarea 3

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split

print("Cargando datos y preparando Tarea 3 (Categorización de Sexismo)...")

# 1. Rutas (Ajusta CARPETA_BASE_IMAGENES si es diferente en tu entorno)
ruta_json = 'EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json'
CARPETA_BASE_IMAGENES = 'EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/'

with open(ruta_json, 'r', encoding='utf-8') as f:
    data_json = json.load(f)

# Defino las clases de la Tarea 3
clases_task3 = [
    "IDEOLOGICAL-INEQUALITY", 
    "STEREOTYPING-DOMINANCE", 
    "OBJECTIFICATION", 
    "SEXUAL-VIOLENCE", 
    "MISOGYNY-NON-SEXUAL-VIOLENCE"
]
lista_datos_t3 = []

# Itero y filtro
for meme_id, info in data_json.items():
    votos_t1 = info.get('labels_task2_1', [])
    votos_yes = votos_t1.count('YES')
    votos_no = votos_t1.count('NO')
    
    # FILTRO: Solo continuamos si la mayoría dijo YES
    if votos_yes > votos_no:
        votos_t3 = info.get('labels_task2_3', [])
        
        # LIMPIEZA: Filtramos los ["-"]
        anotadores_validos = [lista_votos for lista_votos in votos_t3 if not (len(lista_votos) == 1 and lista_votos[0] == "-")]
        total_anotadores = len(anotadores_validos)
        
        if total_anotadores > 0:
            votos_planos = [etiqueta for lista in anotadores_validos for etiqueta in lista if etiqueta in clases_task3]
            counts = {clase: votos_planos.count(clase) for clase in clases_task3}
            
            # Soft Labels: array de 5 posiciones
            soft_label_t3 = [counts[clase] / total_anotadores for clase in clases_task3]
            
            lista_datos_t3.append({
                'id_EXIST': info['id_EXIST'],
                'text': info['text'],
                'path_memes': info['path_memes'],
                'labels_task2_3': votos_t3,
                'soft_label': soft_label_t3,
                'sensorial': info.get('sensorial', {}) 
            })

# Convierto a DF de pandas 
df_completo_t3 = pd.DataFrame(lista_datos_t3)

print(f"Total de memes originales en el JSON: {len(data_json)}")
print(f"Memes SÍ Sexistas listos para la Tarea 3: {len(df_completo_t3)}")

# Particiones
df_train, df_temp = train_test_split(df_completo_t3, test_size=0.20, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=42)

print("\nPartición completada:")
print(f" - Train: {len(df_train)} memes")
print(f" - Val:   {len(df_val)} memes")
print(f" - Test:  {len(df_test)} memes")


# ==========================================
#  CREACIÓN DE DATASETS Y DATALOADERS
# ==========================================
class MemeDataset_T3(Dataset):
    def __init__(self, df, tokenizer, processor, img_dir):
        self.df = df
        self.tokenizer = tokenizer
        self.processor = processor
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Procesar Texto
        text = row['text']
        encoding = self.tokenizer(text, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
        
        # Procesar Imagen
        img_path = os.path.join(self.img_dir, row['path_memes'])
        image = Image.open(img_path).convert("RGB")
        img_encoding = self.processor(images=image, return_tensors="pt")
        
        # ⚠️ AQUÍ ESTÁ EL ARREGLO: Extrae las 5 probabilidades como target
        targets = torch.tensor(row['soft_label'], dtype=torch.float32)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'pixel_values': img_encoding['pixel_values'].squeeze(0),
            'targets': targets
        }

print("\nConstruyendo DataLoaders con targets de dimensión 5...")
train_dataset = MemeDataset_T3(df_train, tokenizer_txt, processor_img, CARPETA_BASE_IMAGENES)
val_dataset = MemeDataset_T3(df_val, tokenizer_txt, processor_img, CARPETA_BASE_IMAGENES)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print(" DataLoaders listos. ¡Ya puedes lanzar el bucle de entrenamiento!")

divido en train, val test

In [ ]:
# Separo el 80% para Train
df_train, df_temp = train_test_split(df_completo, test_size=0.20, random_state=42)

# Reparto entre val y test
df_val, df_test = train_test_split(df_temp, test_size=0.50, random_state=42)

print("Partición completada:")
print(f" - Train: {len(df_train)} memes")
print(f" - Val:   {len(df_val)} memes")
print(f" - Test:  {len(df_test)} memes")

print("\nSoft Labels calculadas. Ejemplo del primer meme en Train:")
print(f"Votos: {df_train['labels_task2_2'].iloc[0]}")
print(f"Soft Label [NO, YES]: {df_train['soft_label'].iloc[0]}")

display(df_train.head())

DATOS EEG

In [ ]:
print("Extrayendo EEG...")

lista_datos_eeg = []

for index, row in df_completo.iterrows():
    id_meme = row['id_EXIST']
    soft_label = row['soft_label']

    sensorial = row.get('sensorial', {})
    tiene_eeg = False

    if isinstance(sensorial, dict):
        modalities = sensorial.get('modalities', {})
        eeg_data = modalities.get('EEG', {}).get('by_user', {})

        if eeg_data:
            tiene_eeg = True
            for user_id, mediciones in eeg_data.items():
                fila = {
                    'id_EXIST': id_meme,
                    'user_id': user_id,
                    'soft_label': soft_label
                }
                for nombre_feature, valor in mediciones.items():
                    fila[nombre_feature] = valor

                lista_datos_eeg.append(fila)

    if not tiene_eeg:
        lista_datos_eeg.append({
            'id_EXIST': id_meme,
            'user_id': 'SIN_EEG',
            'soft_label': soft_label
        })

# Convierto a DataFrame
df_eeg = pd.DataFrame(lista_datos_eeg)

print(f"Dimensiones del dataset EEG: {df_eeg.shape}")
print(f"Valores NaN antes de imputar: {df_eeg.isna().sum().sum()}")

# Limpieza y relleno de NaNs
cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_features = [col for col in df_eeg.columns if col not in cols_metadatos]

df_eeg[cols_features] = df_eeg[cols_features].fillna(df_eeg[cols_features].mean())
df_eeg[cols_features] = df_eeg[cols_features].fillna(0)

print(f"Valores NaN después de imputar: {df_eeg.isna().sum().sum()}")

# Resultado
print("\nResultado final:")
display(df_eeg[['id_EXIST', 'user_id', 'soft_label'] + cols_features[:7]].head(6))

In [ ]:
print("Preparando el Dataset EEG...")

# SPLIT SEGURO
df_eeg_train = df_eeg[df_eeg['id_EXIST'].isin(df_train['id_EXIST'])].reset_index(drop=True)
df_eeg_val   = df_eeg[df_eeg['id_EXIST'].isin(df_val['id_EXIST'])].reset_index(drop=True)
df_eeg_test  = df_eeg[df_eeg['id_EXIST'].isin(df_test['id_EXIST'])].reset_index(drop=True)

print(f"Partición EEG -> Train: {len(df_eeg_train)} | Val: {len(df_eeg_val)} | Test: {len(df_eeg_test)}")

cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_canales_eeg = [col for col in df_eeg.columns if col not in cols_metadatos]

# DEFINICIÓN DEL DATASET
class DatasetEEG_SoftLabels(Dataset):
    def __init__(self, df, cols_features):
        self.df = df
        self.features = df[cols_features].values.astype(np.float32)
        self.labels = np.array(df['soft_label'].tolist(), dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'eeg': torch.tensor(self.features[idx]),
            'targets': torch.tensor(self.labels[idx])
        }

# CREACIÓN DE LOS LOADERS
train_loader_eeg = DataLoader(DatasetEEG_SoftLabels(df_eeg_train, cols_canales_eeg), batch_size=64, shuffle=True)
val_loader_eeg   = DataLoader(DatasetEEG_SoftLabels(df_eeg_val, cols_canales_eeg), batch_size=64, shuffle=False)
test_loader_eeg  = DataLoader(DatasetEEG_SoftLabels(df_eeg_test, cols_canales_eeg), batch_size=64, shuffle=False)

print("DataLoaders de EEG listos para entrenar.")

DATOS ET

In [ ]:
print("Extrayendo datos de Eye-Tracking (ET)...")

lista_datos_et = []

for index, row in df_completo.iterrows():
    id_meme = row['id_EXIST']
    soft_label = row['soft_label']

    sensorial = row.get('sensorial', {})
    tiene_et = False

    if isinstance(sensorial, dict):
        modalities = sensorial.get('modalities', {})
        et_data = modalities.get('ET', {}).get('by_user', {})

        if et_data:
            tiene_et = True
            for user_id, mediciones in et_data.items():
                fila = {
                    'id_EXIST': id_meme,
                    'user_id': user_id,
                    'soft_label': soft_label
                }

                for nombre_feature, valor in mediciones.items():
                    fila[nombre_feature] = valor

                lista_datos_et.append(fila)

    if not tiene_et:
        lista_datos_et.append({
            'id_EXIST': id_meme,
            'user_id': 'SIN_ET',
            'soft_label': soft_label
        })

# Convierto a DataFrame
df_et = pd.DataFrame(lista_datos_et)

print(f"Dimensiones del dataset Eye-Tracking: {df_et.shape}")
print(f"Valores NaN antes de imputar: {df_et.isna().sum().sum()}")

# Limpieza y relleno de NaNs
cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_features_et = [col for col in df_et.columns if col not in cols_metadatos]

df_et[cols_features_et] = df_et[cols_features_et].fillna(df_et[cols_features_et].mean())
df_et[cols_features_et] = df_et[cols_features_et].fillna(0)

print(f"Valores NaN después de imputar: {df_et.isna().sum().sum()}")

# Resultado
print("\nMuestra del Eye-Tracking:")
display(df_et[['id_EXIST', 'user_id', 'soft_label'] + cols_features_et[:7]].head(6))

In [ ]:
print("Preparando el Dataset de Eye-Tracking (ET)...")

# SPLIT SEGURO
df_et_train = df_et[df_et['id_EXIST'].isin(df_train['id_EXIST'])].reset_index(drop=True)
df_et_val   = df_et[df_et['id_EXIST'].isin(df_val['id_EXIST'])].reset_index(drop=True)
df_et_test  = df_et[df_et['id_EXIST'].isin(df_test['id_EXIST'])].reset_index(drop=True)

print(f"Partición ET -> Train: {len(df_et_train)} | Val: {len(df_et_val)} | Test: {len(df_et_test)}")

# Extraigo las columnas que corresponden a las métricas de la mirada
cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_features_et = [col for col in df_et.columns if col not in cols_metadatos]

# DEFINICIÓN DEL DATASET ET
class DatasetET_SoftLabels(Dataset):
    def __init__(self, df, cols_features):
        self.df = df

        self.features = df[cols_features].values.astype(np.float32)
        self.labels = np.array(df['soft_label'].tolist(), dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'et': torch.tensor(self.features[idx]),
            'targets': torch.tensor(self.labels[idx])
        }

#  CREACIÓN DE LOS LOADERS
train_loader_et = DataLoader(DatasetET_SoftLabels(df_et_train, cols_features_et), batch_size=64, shuffle=True)
val_loader_et   = DataLoader(DatasetET_SoftLabels(df_et_val, cols_features_et), batch_size=64, shuffle=False)
test_loader_et  = DataLoader(DatasetET_SoftLabels(df_et_test, cols_features_et), batch_size=64, shuffle=False)

print("DataLoaders de Eye-Tracking (ET) listos para entrenar.")

DATOS HR

In [ ]:
print("Extrayendo datos de Heart Rate (HR)...")

lista_datos_hr = []

for index, row in df_completo.iterrows():
    id_meme = row['id_EXIST']
    soft_label = row['soft_label']

    sensorial = row.get('sensorial', {})
    tiene_hr = False

    if isinstance(sensorial, dict):
        modalities = sensorial.get('modalities', {})
        hr_data = modalities.get('HR', {}).get('by_user', {})

        if hr_data:
            tiene_hr = True
            for user_id, mediciones in hr_data.items():
                fila = {
                    'id_EXIST': id_meme,
                    'user_id': user_id,
                    'soft_label': soft_label
                }

                for nombre_feature, valor in mediciones.items():
                    fila[nombre_feature] = valor

                lista_datos_hr.append(fila)

    if not tiene_hr:
        lista_datos_hr.append({
            'id_EXIST': id_meme,
            'user_id': 'SIN_HR',
            'soft_label': soft_label
        })

# Convierto a DataFrame
df_hr = pd.DataFrame(lista_datos_hr)

print(f"Dimensiones del dataset Heart Rate: {df_hr.shape}")
print(f"Valores NaN antes de imputar: {df_hr.isna().sum().sum()}")

# Limpieza y relleno de NaNs
cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_features_hr = [col for col in df_hr.columns if col not in cols_metadatos]

df_hr[cols_features_hr] = df_hr[cols_features_hr].fillna(df_hr[cols_features_hr].mean())
df_hr[cols_features_hr] = df_hr[cols_features_hr].fillna(0)

print(f"Valores NaN después de imputar: {df_hr.isna().sum().sum()}")

# Resultado
print("\n Muestra de Heart Rate (HR):")
display(df_hr[['id_EXIST', 'user_id', 'soft_label'] + cols_features_hr[:4]].head(6))

In [ ]:
print("Preparando el Dataset de Heart Rate (HR)...")

# SPLIT SEGURO
df_hr_train = df_hr[df_hr['id_EXIST'].isin(df_train['id_EXIST'])].reset_index(drop=True)
df_hr_val   = df_hr[df_hr['id_EXIST'].isin(df_val['id_EXIST'])].reset_index(drop=True)
df_hr_test  = df_hr[df_hr['id_EXIST'].isin(df_test['id_EXIST'])].reset_index(drop=True)

print(f"Partición HR -> Train: {len(df_hr_train)} | Val: {len(df_hr_val)} | Test: {len(df_hr_test)}")

# Extraigo las columnas que corresponden a las métricas del ritmo cardíaco
cols_metadatos = ['id_EXIST', 'user_id', 'soft_label']
cols_features_hr = [col for col in df_hr.columns if col not in cols_metadatos]

# DEFINICIÓN DEL DATASET HR
class DatasetHR_SoftLabels(Dataset):
    def __init__(self, df, cols_features):
        self.df = df
        # Extraigo todas las métricas del ritmo cardíaco
        self.features = df[cols_features].values.astype(np.float32)
        # Extraigo las Soft Labels
        self.labels = np.array(df['soft_label'].tolist(), dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'hr': torch.tensor(self.features[idx]), # <-- Cambiamos la clave a 'hr'
            'targets': torch.tensor(self.labels[idx])
        }

# CREACIÓN DE LOS LOADERS
train_loader_hr = DataLoader(DatasetHR_SoftLabels(df_hr_train, cols_features_hr), batch_size=64, shuffle=True)
val_loader_hr   = DataLoader(DatasetHR_SoftLabels(df_hr_val, cols_features_hr), batch_size=64, shuffle=False)
test_loader_hr  = DataLoader(DatasetHR_SoftLabels(df_hr_test, cols_features_hr), batch_size=64, shuffle=False)

print("DataLoaders de Heart Rate (HR) listos para entrenar.")

# Carga del modelo

In [ ]:
print("Cargando Tokenizador (RoBERTa) y Procesador (CLIP)...")
tokenizer_txt = AutoTokenizer.from_pretrained("cardiffnlp/twitter-xlm-roberta-base")
processor_img = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class DatasetSoftLabelsDual(Dataset):
    def __init__(self, df, tokenizer, processor, base_img_path='EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/'):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.processor = processor
        self.base_img_path = base_img_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- LENGUAJE (RoBERTa) ---
        texto = str(row['text'])
        inputs_txt = self.tokenizer(
            texto,
            return_tensors="pt",
            padding="max_length",
            max_length=128,
            truncation=True
        )

        # --- VISIÓN (CLIP) ---
        ruta_imagen = os.path.join(self.base_img_path, row['path_memes'])
        image = Image.open(ruta_imagen).convert("RGB")
        inputs_img = self.processor(images=image, return_tensors="pt")

        # --- ETIQUETAS (Soft Labels) ---
        target_probs = torch.tensor(row['soft_label'], dtype=torch.float32)

        return {
            'input_ids': inputs_txt['input_ids'].squeeze(0),
            'attention_mask': inputs_txt['attention_mask'].squeeze(0),
            'pixel_values': inputs_img['pixel_values'].squeeze(0),
            'targets': target_probs
        }

# Creo los Loaders
train_loader = DataLoader(DatasetSoftLabelsDual(df_train, tokenizer_txt, processor_img), batch_size=32, shuffle=True)
val_loader = DataLoader(DatasetSoftLabelsDual(df_val, tokenizer_txt, processor_img), batch_size=32, shuffle=False)
test_loader = DataLoader(DatasetSoftLabelsDual(df_test, tokenizer_txt, processor_img), batch_size=32, shuffle=False)

print(f"Datos empaquetados. Batches de Train: {len(train_loader)}")

In [ ]:
# Clase para el modelo hibrido
class TorreHibrida_SoftLabels(nn.Module):
    def __init__(self):
        super().__init__()

        # Visión (Da un vector de 768)
        self.vision_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        # Lenguaje (Da un vector de 768)
        self.text_model = AutoModel.from_pretrained("cardiffnlp/twitter-xlm-roberta-base")

        # Congelar el modelo de Visión
        #for param in self.vision_model.parameters():
            #param.requires_grad = False

        # Congelar el modelo de Lenguaje
        #for param in self.text_model.parameters():
            #param.requires_grad = False

        # 3. Clasificador (768 + 768 = 1536)
        self.clasificador = nn.Sequential(
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2) # Salida de 2 neuronas (Logits para NO y YES)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # Visión
        vision_outputs = self.vision_model(pixel_values=pixel_values)
        img_embeds = vision_outputs.pooler_output

        # Lenguaje
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        txt_embeds = text_outputs.last_hidden_state[:, 0, :]

        # Fusión
        combined = torch.cat((img_embeds, txt_embeds), dim=1)

        return self.clasificador(combined)

# Construimos el modelo y lo mandamos a la gráfica
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_vl_soft = TorreHibrida_SoftLabels().to(device)

print(f"Modelo Hibrido construido")

tarea 3

In [ ]:
# Clase para el modelo hibrido
class TorreHibrida_SoftLabels(nn.Module):
    def __init__(self):
        super().__init__()

        # Visión (Da un vector de 768)
        self.vision_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        # Lenguaje (Da un vector de 768)
        self.text_model = AutoModel.from_pretrained("cardiffnlp/twitter-xlm-roberta-base")

        # Congelar el modelo de Visión
        for param in self.vision_model.parameters():
            param.requires_grad = False

        # Congelar el modelo de Lenguaje
        for param in self.text_model.parameters():
            param.requires_grad = False

        # 3. Clasificador (768 + 768 = 1536)
        self.clasificador = nn.Sequential(
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 5) # Salida de 5 neuronas (Una por cada tipo)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # Visión
        vision_outputs = self.vision_model(pixel_values=pixel_values)
        img_embeds = vision_outputs.pooler_output

        # Lenguaje
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        txt_embeds = text_outputs.last_hidden_state[:, 0, :]

        # Fusión
        combined = torch.cat((img_embeds, txt_embeds), dim=1)

        return self.clasificador(combined)

# Construimos el modelo y lo mandamos a la gráfica
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_vl_t3 = TorreHibrida_SoftLabels().to(device)

print(f"Modelo Hibrido construido")

EEG

In [ ]:
# ARQUITECTURA DEL MODELO EEG
class TorreEEG_SoftLabels(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.red(x)

# Instancio el modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dimension = len(cols_canales_eeg) # Debería ser 80
modelo_eeg_soft = TorreEEG_SoftLabels(input_dim=input_dimension).to(device)

print(f"Torre EEG construida con {input_dimension} entradas.")

# 3. Lo convertimos a un tensor de PyTorch y lo mandamos a la gráfica
pesos_tensor = torch.tensor([1.0, 1.5], dtype=torch.float32).to(device)

print(f"- Pesos calculados automáticamente: NO={pesos_tensor[0]:.4f}, YES={pesos_tensor[1]:.4f}")

# 4. Se lo inyectamos a la función de pérdida
criterion = nn.CrossEntropyLoss(weight=pesos_tensor)

# CONFIGURACIÓN DEL ENTRENAMIENTO
optimizer_eeg = optim.AdamW(modelo_eeg_soft.parameters(), lr=1e-3, weight_decay=0.05)
#criterion = nn.CrossEntropyLoss()

mejor_val_loss_eeg = float('inf')
epocas_eeg = 20

print("\nINICIANDO ENTRENAMIENTO DE LA TORRE EEG...")

for epoch in range(epocas_eeg):
    # --- TRAIN ---
    modelo_eeg_soft.train()
    train_loss = 0.0

    for batch in train_loader_eeg:
        eeg_data = batch['eeg'].to(device)
        targets = batch['targets'].to(device)

        optimizer_eeg.zero_grad()
        logits = modelo_eeg_soft(eeg_data)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer_eeg.step()

        train_loss += loss.item()

    train_loss_media = train_loss / len(train_loader_eeg)

    # --- VAL ---
    modelo_eeg_soft.eval()
    val_loss = 0.0
    preds_duras = []
    reales_duras = []

    with torch.no_grad():
        for batch in val_loader_eeg:
            eeg_data = batch['eeg'].to(device)
            targets = batch['targets'].to(device)

            logits = modelo_eeg_soft(eeg_data)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            # F1 Simulado para seguimiento
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            reales_batch = torch.argmax(targets, dim=1).cpu().numpy()
            preds_duras.extend(preds_batch)
            reales_duras.extend(reales_batch)

    val_loss_media = val_loss / len(val_loader_eeg)
    f1_val = f1_score(reales_duras, preds_duras, average='macro', zero_division=0)

    # Imprimo progreso cada 2 épocas para no llenar la pantalla
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"- Época {epoch+1:02d}/{epocas_eeg} | Train Loss: {train_loss_media:.4f} | Val Loss: {val_loss_media:.4f} | Val F1: {f1_val:.4f}")

    # Guardo el mejor
    if val_loss_media < mejor_val_loss_eeg:
        mejor_val_loss_eeg = val_loss_media
        torch.save(modelo_eeg_soft.state_dict(), "mejor_modelo_eeg_soft.pth")

print("ENTRENAMIENTO EEG FINALIZADO. Mejor modelo guardado.")

ET

In [ ]:
# ARQUITECTURA DEL MODELO ET
class TorreET_SoftLabels(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.red(x)

# Instancio el modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dimension_et = len(cols_features_et)
modelo_et_soft = TorreET_SoftLabels(input_dim=input_dimension_et).to(device)

print(f"Torre ET construida con {input_dimension_et} entradas.")
pesos_tensor = torch.tensor([1.0, 1.8], dtype=torch.float32).to(device)

print(f"- Pesos calculados automáticamente: NO={pesos_tensor[0]:.4f}, YES={pesos_tensor[1]:.4f}")

#  Se lo inyectamos a la función de pérdida
criterion = nn.CrossEntropyLoss(weight=pesos_tensor)

#  CONFIGURACIÓN DEL ENTRENAMIENTO
optimizer_et = optim.AdamW(modelo_et_soft.parameters(), lr=3e-4, weight_decay=0.01)
#criterion = nn.CrossEntropyLoss()

mejor_val_loss_et = float('inf')
epocas_et = 35

print("\nINICIANDO ENTRENAMIENTO DE LA TORRE ET...")

for epoch in range(epocas_et):
    # --- TRAIN ---
    modelo_et_soft.train()
    train_loss = 0.0

    for batch in train_loader_et:
        et_data = batch['et'].to(device)
        targets = batch['targets'].to(device)

        optimizer_et.zero_grad()
        logits = modelo_et_soft(et_data)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer_et.step()

        train_loss += loss.item()

    train_loss_media = train_loss / len(train_loader_et)

    # --- VAL ---
    modelo_et_soft.eval()
    val_loss = 0.0
    preds_duras = []
    reales_duras = []

    with torch.no_grad():
        for batch in val_loader_et:
            et_data = batch['et'].to(device)
            targets = batch['targets'].to(device)

            logits = modelo_et_soft(et_data)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            # F1 Simulado para seguimiento
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            reales_batch = torch.argmax(targets, dim=1).cpu().numpy()
            preds_duras.extend(preds_batch)
            reales_duras.extend(reales_batch)

    val_loss_media = val_loss / len(val_loader_et)
    f1_val = f1_score(reales_duras, preds_duras, average='macro', zero_division=0)

    # Imprimo progreso cada 2 épocas para no llenar la pantalla
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"- Época {epoch+1:02d}/{epocas_et} | Train Loss: {train_loss_media:.4f} | Val Loss: {val_loss_media:.4f} | Val F1: {f1_val:.4f}")

    # Guardo el mejor
    if val_loss_media < mejor_val_loss_et:
        mejor_val_loss_et = val_loss_media
        torch.save(modelo_et_soft.state_dict(), "mejor_modelo_et_soft.pth")

print("ENTRENAMIENTO ET FINALIZADO. Mejor modelo guardado.")

HR

In [ ]:
# ARQUITECTURA DEL MODELO HR
class TorreHR_SoftLabels(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.red(x)

# Instancio el modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dimension_hr = len(cols_features_hr)
modelo_hr_soft = TorreHR_SoftLabels(input_dim=input_dimension_hr).to(device)

print(f"Torre HR construida con {input_dimension_hr} entradas.")
pesos_tensor = torch.tensor([1.0, 1.8], dtype=torch.float32).to(device)

print(f"- Pesos calculados automáticamente: NO={pesos_tensor[0]:.4f}, YES={pesos_tensor[1]:.4f}")

# 4. Se lo inyectamos a la función de pérdida
criterion = nn.CrossEntropyLoss(weight=pesos_tensor)

# CONFIGURACIÓN DEL ENTRENAMIENTO
optimizer_hr = optim.AdamW(modelo_hr_soft.parameters(), lr=3e-4, weight_decay=0.01)
#criterion = nn.CrossEntropyLoss()

mejor_val_loss_hr = float('inf')
epocas_hr = 35

print("\nINICIANDO ENTRENAMIENTO DE LA TORRE HR...")

for epoch in range(epocas_hr):
    # --- TRAIN ---
    modelo_hr_soft.train()
    train_loss = 0.0

    for batch in train_loader_hr:
        hr_data = batch['hr'].to(device)
        targets = batch['targets'].to(device)

        optimizer_hr.zero_grad()
        logits = modelo_hr_soft(hr_data)

        loss = criterion(logits, targets)
        loss.backward()
        optimizer_hr.step()

        train_loss += loss.item()

    train_loss_media = train_loss / len(train_loader_hr)

    # --- VAL ---
    modelo_hr_soft.eval()
    val_loss = 0.0
    preds_duras = []
    reales_duras = []

    with torch.no_grad():
        for batch in val_loader_hr:
            hr_data = batch['hr'].to(device)
            targets = batch['targets'].to(device)

            logits = modelo_hr_soft(hr_data)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            # F1 Simulado para seguimiento
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            reales_batch = torch.argmax(targets, dim=1).cpu().numpy()
            preds_duras.extend(preds_batch)
            reales_duras.extend(reales_batch)

    val_loss_media = val_loss / len(val_loader_hr)
    f1_val = f1_score(reales_duras, preds_duras, average='macro', zero_division=0)

    # Imprimo progreso cada 2 épocas para no llenar la pantalla
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"- Época {epoch+1:02d}/{epocas_hr} | Train Loss: {train_loss_media:.4f} | Val Loss: {val_loss_media:.4f} | Val F1: {f1_val:.4f}")

    # Guardo el mejor
    if val_loss_media < mejor_val_loss_hr:
        mejor_val_loss_hr = val_loss_media
        torch.save(modelo_hr_soft.state_dict(), "mejor_modelo_hr_soft.pth")

print("ENTRENAMIENTO HR FINALIZADO. Mejor modelo guardado.")

# Entrenamiento

In [ ]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

print(" INICIANDO ENTRENAMIENTO...")

# Configuración del Optimizador y la Función de Pérdida
optimizer = torch.optim.AdamW(modelo_vl_soft.parameters(), lr=1e-6, weight_decay=0.005)

# Para ver cuanto acierta en los soft labes
#criterion = nn.CrossEntropyLoss()

pesos_clases = torch.tensor([1.0, 1.8]).to(device) 

# Le pasamos los pesos a la función de pérdida
criterion = nn.CrossEntropyLoss(weight=pesos_clases)

mejor_val_loss = float('inf')
epocas = 5

for epoch in range(epocas):
    # ================================
    #  ENTRENAMIENTO
    # ================================
    modelo_vl_soft.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Época {epoch+1}/{epocas} - Train"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        targets = batch['targets'].to(device)

        optimizer.zero_grad()

        # Predicción del modelo
        logits = modelo_vl_soft(pixel_values, input_ids, attention_mask)

        # Cálculo del error y retropropagación
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss_media = train_loss / len(train_loader)

    # ================================
    # VALIDACIÓN
    # ================================
    modelo_vl_soft.eval()
    val_loss = 0.0

    preds_duras = []
    reales_duras = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Época {epoch+1}/{epocas} - Val"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            targets = batch['targets'].to(device)

            logits = modelo_vl_soft(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            # argmax saca el índice de la probabilidad más alta (0 para NO, 1 para YES)
            preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
            reales_batch = torch.argmax(targets, dim=1).cpu().numpy()

            preds_duras.extend(preds_batch)
            reales_duras.extend(reales_batch)

    val_loss_media = val_loss / len(val_loader)
    f1_val = f1_score(reales_duras, preds_duras, average='macro')

    print(f"- Época {epoch+1} | Train Loss: {train_loss_media:.4f} | Val Loss: {val_loss_media:.4f} | Val F1 (Simulado): {f1_val:.4f}")

    # ================================
    # GUARDAR EL MEJOR
    # ================================
    if val_loss_media < mejor_val_loss:
        mejor_val_loss = val_loss_media
        torch.save(modelo_vl_soft.state_dict(), "mejor_modelo_vl_soft.pth")
        print("   ¡Nuevo mejor modelo guardado!")

print("\nENTRENAMIENTO FINALIZADO.")

Tarea 3

In [ ]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import torch
import torch.nn as nn

print(" INICIANDO ENTRENAMIENTO MULTI-ETIQUETA (TAREA 3)...")

print("Calculando pesos (pos_weight) para equilibrar las clases...")

# Extraemos todas las soft labels de entrenamiento y las convertimos en matriz
etiquetas_train = np.array(df_train['soft_label'].tolist())

# Contamos cuántos positivos (mayor al 50%) y negativos hay por cada clase
# axis=0 suma las columnas hacia abajo
positivos_por_clase = (etiquetas_train > 0.5).sum(axis=0)
total_muestras = len(etiquetas_train)
negativos_por_clase = total_muestras - positivos_por_clase

#  Aplicamos la fórmula matemática: Negativos / Positivos
# Usamos np.maximum(..., 1) para evitar que el programa explote dividiendo por cero
pesos_positivos = negativos_por_clase / np.maximum(positivos_por_clase, 1)

# Lo convertimos a tensor de PyTorch y lo mandamos a la tarjeta gráfica
pos_weight_tensor = torch.tensor(pesos_positivos, dtype=torch.float32).to(device)

nombres_clases = ["Ideo", "Stereo", "Object", "SexVio", "Miso"]
print("- Multiplicadores de castigo calculados:")
for nombre, peso in zip(nombres_clases, pesos_positivos):
    print(f"   - {nombre}: castigo x{peso:.2f}")

#  INYECTO PESOS EN LA FUNCIÓN DE PÉRDIDA
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# Usamos el modelo que creamos para la Tarea 3
optimizer = torch.optim.AdamW(modelo_vl_t3.parameters(), lr=1e-4, weight_decay=0.01)


mejor_val_loss = float('inf')
epocas = 10 # Ponle unas pocas más porque es una tarea más difícil

for epoch in range(epocas):
    # ================================
    #  ENTRENAMIENTO
    # ================================
    modelo_vl_t3.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Época {epoch+1}/{epocas} - Train"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        # targets ahora es un vector de 5 posiciones [0.66, 0.0, 0.33, 0.0, 0.0]
        targets = batch['targets'].to(device).float() 

        optimizer.zero_grad()

        # Predicción del modelo
        logits = modelo_vl_t3(pixel_values, input_ids, attention_mask)

        # Cálculo del error y retropropagación
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss_media = train_loss / len(train_loader)

    # ================================
    # VALIDACIÓN
    # ================================
    modelo_vl_t3.eval()
    val_loss = 0.0

    preds_duras = []
    reales_duras = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Época {epoch+1}/{epocas} - Val"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            pixel_values = batch['pixel_values'].to(device)
            targets = batch['targets'].to(device).float()

            logits = modelo_vl_t3(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            #  Evaluación Multi-Etiqueta
            # Pasamos los logits por la Sigmoide para sacar porcentajes del 0 al 1
            probs = torch.sigmoid(logits)
            
            # Si supera el 50% de probabilidad, decimos que SÍ tiene esa etiqueta (1), si no (0)
            preds_batch = (probs > 0.5).int().cpu().numpy()
            
            # Para calcular el F1, asumimos que si el target soft supera el 50%, era la clase "real" mayoritaria
            reales_batch = (targets > 0.5).int().cpu().numpy()

            preds_duras.extend(preds_batch)
            reales_duras.extend(reales_batch)

    val_loss_media = val_loss / len(val_loader)
    
    # average='macro' funciona perfecto para matrices multi-etiqueta
    f1_val = f1_score(reales_duras, preds_duras, average='macro', zero_division=0)

    print(f"- Época {epoch+1} | Train Loss: {train_loss_media:.4f} | Val Loss: {val_loss_media:.4f} | Val F1: {f1_val:.4f}")

    # ================================
    # GUARDAR EL MEJOR
    # ================================
    if val_loss_media < mejor_val_loss:
        mejor_val_loss = val_loss_media
        torch.save(modelo_vl_t3.state_dict(), "mejor_modelo_vl_t3.pth")
        print("   ¡Nuevo mejor modelo T3 guardado!")

print("\nENTRENAMIENTO FINALIZADO.")

# Resultado

In [ ]:
print("INICIANDO EXAMEN EN EL CONJUNTO DE TEST...")

# Cargo los pesos del mejor modelo que se guardó en la validación
modelo_vl_soft.load_state_dict(torch.load("mejor_modelo_vl_soft.pth", weights_only=True))
modelo_vl_soft.eval()

test_preds = []
test_reales = []

# Ejecuto el examen
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Analizando Memes de Test"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        targets = batch['targets'].to(device)


        logits = modelo_vl_soft(pixel_values, input_ids, attention_mask)

        # TRADUCCIÓN A DECISIÓN FINAL
        # argmax elige el índice del valor más alto: 0 para NO, 1 para YES
        preds_batch = torch.argmax(logits, dim=1).cpu().numpy()

        reales_batch = torch.argmax(targets, dim=1).cpu().numpy()

        test_preds.extend(preds_batch)
        test_reales.extend(reales_batch)

# RESULTADOS NUMÉRICOS
nombres_clases = ['NO Sexista (0)', 'SÍ Sexista (1)']

print("\n" + "="*50)
print(" INFORME DETALLADO DE MÉTRICAS (TEST)")
print("="*50)
print(classification_report(test_reales, test_preds, target_names=nombres_clases, digits=4))

#  MATRIZ DE CONFUSIÓN VISUAL
cm = confusion_matrix(test_reales, test_preds)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases,
            annot_kws={"size": 14})

plt.title('Matriz de Confusión - Vision-Lenguaje Soft Labels', fontsize=14, pad=15)
plt.xlabel('Predicción', fontsize=12, labelpad=10)
plt.ylabel('Realidad', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

TAREA 3

In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import numpy as np

test_dataset = MemeDataset_T3(df_test, tokenizer_txt, processor_img, CARPETA_BASE_IMAGENES)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
print("INICIANDO EXAMEN EN EL CONJUNTO DE TEST (TAREA 3)...")

# Cargamos los pesos del mejor modelo de la Tarea 3
# Asegúrate de usar la variable correcta de tu modelo (modelo_vl_t3)
modelo_vl_t3.load_state_dict(torch.load("mejor_modelo_vl_t3.pth", weights_only=True))
modelo_vl_t3.eval()

test_preds = []
test_reales = []

# Ejecuto el examen
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Analizando Memes de Test T3"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device)
        # targets en T3 son 5 probabilidades [0.2, 0.8, 0.0, ...]
        targets = batch['targets'].to(device).float()

        logits = modelo_vl_t3(pixel_values, input_ids, attention_mask)

        # TRADUCCIÓN A DECISIÓN MULTI-ETIQUETA
        # Pasamos por sigmoide y marcamos 1 si supera el 50%, si no 0
        probs = torch.sigmoid(logits)
        preds_batch = (probs > 0.5).int().cpu().numpy()
        
        # Para saber la "realidad", asumimos que si el anotador le dio > 50%, es válido
        reales_batch = (targets > 0.5).int().cpu().numpy()

        test_preds.extend(preds_batch)
        test_reales.extend(reales_batch)

# Convertimos a arrays de numpy para sklearn
test_preds = np.array(test_preds)
test_reales = np.array(test_reales)

# RESULTADOS NUMÉRICOS
nombres_clases = [
    "Ideo-Inequality", 
    "Stereotyping", 
    "Objectification", 
    "Sexual-Violence", 
    "Misogyny-Non-Sexual"
]

print("\n" + "="*60)
print(" INFORME DETALLADO DE MÉTRICAS MULTI-ETIQUETA (TEST)")
print("="*60)
# classification_report es lo suficientemente listo como para entender arrays de 5 columnas
print(classification_report(test_reales, test_preds, target_names=nombres_clases, digits=4, zero_division=0))

# 3. MATRICES DE CONFUSIÓN VISUALES (Una por categoría)
# Genera 5 matrices de 2x2
mcm = multilabel_confusion_matrix(test_reales, test_preds)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Matrices de Confusión Independientes - Tarea 3', fontsize=16, y=1.05)

for i, (ax, matrix) in enumerate(zip(axes, mcm)):
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Purples',
                xticklabels=['NO', 'SÍ'],
                yticklabels=['NO', 'SÍ'],
                annot_kws={"size": 14}, ax=ax, cbar=False)
    
    ax.set_title(nombres_clases[i], fontsize=12, pad=10)
    ax.set_xlabel('Predicción', fontsize=10)
    if i == 0:
        ax.set_ylabel('Realidad', fontsize=10)

plt.tight_layout()
plt.show()

EEG

In [ ]:
print("INICIANDO TEST EEG...")

# Cargo el mejor modelo eeg
modelo_eeg_soft.load_state_dict(torch.load("mejor_modelo_eeg_soft.pth", map_location=device, weights_only=True))
modelo_eeg_soft.eval()

test_preds_eeg = []
test_reales_eeg = []

# Paso el examen
with torch.no_grad():
    for batch in tqdm(test_loader_eeg, desc="Analizando Cerebros de Test"):
        eeg_data = batch['eeg'].to(device)
        targets = batch['targets'].to(device)

        # El modelo EEG da sus Logits
        logits = modelo_eeg_soft(eeg_data)

        # TRADUCCIÓN A DECISIÓN FINAL (Hard Labels)
        preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
        reales_batch = torch.argmax(targets, dim=1).cpu().numpy()

        test_preds_eeg.extend(preds_batch)
        test_reales_eeg.extend(reales_batch)

# RESULTADOS NUMÉRICOS
nombres_clases = ['NO Sexista (0)', 'SÍ Sexista (1)']

print("\n" + "="*50)
print("INFORME DETALLADO DE MÉTRICAS -  EEG")
print("="*50)
print(classification_report(test_reales_eeg, test_preds_eeg, target_names=nombres_clases, digits=4, zero_division=0))

# MATRIZ DE CONFUSIÓN
cm_eeg = confusion_matrix(test_reales_eeg, test_preds_eeg)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_eeg, annot=True, fmt='d', cmap='Oranges',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases,
            annot_kws={"size": 14})

plt.title('Matriz de Confusión - Solo EEG (Soft Labels)', fontsize=14, pad=15)
plt.xlabel('Predicción Modelo', fontsize=12, labelpad=10)
plt.ylabel('Realidad (Voto Mayoritario)', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

ET

In [ ]:
print("INICIANDO TEST ET...")

# Cargo el mejor modelo ET
modelo_et_soft.load_state_dict(torch.load("mejor_modelo_et_soft.pth", map_location=device, weights_only=True))
modelo_et_soft.eval()

test_preds_et = []
test_reales_et = []

# Paso el examen
with torch.no_grad():
    for batch in tqdm(test_loader_et, desc="Analizando Miradas de Test"):
        et_data = batch['et'].to(device)
        targets = batch['targets'].to(device)

        # El modelo ET da sus Logits
        logits = modelo_et_soft(et_data)

        # TRADUCCIÓN A DECISIÓN FINAL (Hard Labels)
        preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
        reales_batch = torch.argmax(targets, dim=1).cpu().numpy()

        test_preds_et.extend(preds_batch)
        test_reales_et.extend(reales_batch)

# RESULTADOS NUMÉRICOS
nombres_clases = ['NO Sexista (0)', 'SÍ Sexista (1)']

print("\n" + "="*50)
print("INFORME DETALLADO DE MÉTRICAS - ET")
print("="*50)
print(classification_report(test_reales_et, test_preds_et, target_names=nombres_clases, digits=4, zero_division=0))

# MATRIZ DE CONFUSIÓN
cm_et = confusion_matrix(test_reales_et, test_preds_et)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_et, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases,
            annot_kws={"size": 14})

plt.title('Matriz de Confusión - Solo ET (Soft Labels)', fontsize=14, pad=15)
plt.xlabel('Predicción Modelo (Mirada)', fontsize=12, labelpad=10)
plt.ylabel('Realidad (Voto Mayoritario)', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

HR

In [ ]:
print("INICIANDO TEST HR (Heart Rate)...")

# Cargo el mejor modelo HR
modelo_hr_soft.load_state_dict(torch.load("mejor_modelo_hr_soft.pth", map_location=device, weights_only=True))
modelo_hr_soft.eval()

test_preds_hr = []
test_reales_hr = []

# Paso el examen
with torch.no_grad():
    for batch in tqdm(test_loader_hr, desc="Analizando Ritmo Cardíaco de Test"):
        hr_data = batch['hr'].to(device)
        targets = batch['targets'].to(device)

        # El modelo HR da sus Logits
        logits = modelo_hr_soft(hr_data)

        # TRADUCCIÓN A DECISIÓN FINAL (Hard Labels)
        preds_batch = torch.argmax(logits, dim=1).cpu().numpy()
        reales_batch = torch.argmax(targets, dim=1).cpu().numpy()

        test_preds_hr.extend(preds_batch)
        test_reales_hr.extend(reales_batch)

# RESULTADOS NUMÉRICOS
nombres_clases = ['NO Sexista (0)', 'SÍ Sexista (1)']

print("\n" + "="*50)
print(" INFORME DETALLADO DE MÉTRICAS - HR")
print("="*50)
print(classification_report(test_reales_hr, test_preds_hr, target_names=nombres_clases, digits=4, zero_division=0))

# MATRIZ DE CONFUSIÓN
cm_hr = confusion_matrix(test_reales_hr, test_preds_hr)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_hr, annot=True, fmt='d', cmap='Reds',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases,
            annot_kws={"size": 14})

plt.title('Matriz de Confusión - Solo HR (Soft Labels)', fontsize=14, pad=15)
plt.xlabel('Predicción Modelo (Corazón)', fontsize=12, labelpad=10)
plt.ylabel('Realidad (Voto Mayoritario)', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

# RESULTADO FINAL

In [ ]:
print("INICIANDO EL TEST DE FUSIÓN MULTIMODAL (4 MODELOS)")

# Pesos de la Fusión (Deben sumar 1.0)
W_VL = 0.70  # Visión-Lenguaje
W_EEG = 0.20 # Cerebro
W_ET = 0.05  # Mirada
W_HR = 0.05  # Corazón

# Ponemos todos los modelos en modo evaluación
modelo_vl_soft.eval()
modelo_eeg_soft.eval()
modelo_et_soft.eval()
modelo_hr_soft.eval()

test_reales_fusion = []
test_preds_fusion = []

# --- Lista para guardar los datos de la tabla ---
lista_resultados_tabla = []

print(f"Fusionando predicciones para {len(df_test)} memes...")

# Bucle robusto meme a meme
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    id_meme = row['id_EXIST']

    # ==========================================
    # PREDICCIÓN DE VISIÓN-LENGUAJE (VL)
    # ==========================================
    texto = str(row['text'])
    inputs_txt = tokenizer_txt(texto, return_tensors="pt", padding="max_length", max_length=128, truncation=True)
    ruta_imagen = os.path.join('EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/', row['path_memes'])
    image = Image.open(ruta_imagen).convert("RGB")
    inputs_img = processor_img(images=image, return_tensors="pt")

    inputs_txt = {k: v.to(device) for k, v in inputs_txt.items()}
    pixel_values = inputs_img['pixel_values'].to(device)

    with torch.no_grad():
        logits_vl = modelo_vl_soft(pixel_values, inputs_txt['input_ids'], inputs_txt['attention_mask'])
        prob_vl = F.softmax(logits_vl, dim=1).cpu().numpy()[0]

    # ==========================================
    # PREDICCIÓN DE EEG
    # ==========================================
    filas_eeg = df_eeg_test[df_eeg_test['id_EXIST'] == id_meme]
    prob_eeg_media = np.array([0.5, 0.5]) # Duda por defecto

    if len(filas_eeg) > 0 and filas_eeg.iloc[0]['user_id'] != 'SIN_EEG':
        probs_eeg_meme = []
        for _, fila_eeg in filas_eeg.iterrows():
            eeg_tensor = torch.tensor(fila_eeg[cols_canales_eeg].values.astype(np.float32)).unsqueeze(0).to(device)
            with torch.no_grad():
                logits_eeg = modelo_eeg_soft(eeg_tensor)
                prob_eeg = F.softmax(logits_eeg, dim=1).cpu().numpy()[0]
                probs_eeg_meme.append(prob_eeg)
        prob_eeg_media = np.mean(probs_eeg_meme, axis=0)

    # ==========================================
    # PREDICCIÓN DE EYE-TRACKING (ET)
    # ==========================================
    filas_et = df_et_test[df_et_test['id_EXIST'] == id_meme]
    prob_et_media = np.array([0.5, 0.5])

    if len(filas_et) > 0 and filas_et.iloc[0]['user_id'] != 'SIN_ET':
        probs_et_meme = []
        for _, fila_et in filas_et.iterrows():
            et_tensor = torch.tensor(fila_et[cols_features_et].values.astype(np.float32)).unsqueeze(0).to(device)
            with torch.no_grad():
                logits_et = modelo_et_soft(et_tensor)
                prob_et = F.softmax(logits_et, dim=1).cpu().numpy()[0]
                probs_et_meme.append(prob_et)
        prob_et_media = np.mean(probs_et_meme, axis=0)

    # ==========================================
    # PREDICCIÓN DE HEART RATE (HR)
    # ==========================================
    filas_hr = df_hr_test[df_hr_test['id_EXIST'] == id_meme]
    prob_hr_media = np.array([0.5, 0.5])

    if len(filas_hr) > 0 and filas_hr.iloc[0]['user_id'] != 'SIN_HR':
        probs_hr_meme = []
        for _, fila_hr in filas_hr.iterrows():
            hr_tensor = torch.tensor(fila_hr[cols_features_hr].values.astype(np.float32)).unsqueeze(0).to(device)
            with torch.no_grad():
                logits_hr = modelo_hr_soft(hr_tensor)
                prob_hr = F.softmax(logits_hr, dim=1).cpu().numpy()[0]
                probs_hr_meme.append(prob_hr)
        prob_hr_media = np.mean(probs_hr_meme, axis=0)

    # ==========================================
    # FUSIÓN Y DECISIÓN FINAL
    # ==========================================
    prob_final = (W_VL * prob_vl) + (W_EEG * prob_eeg_media) + (W_ET * prob_et_media) + (W_HR * prob_hr_media)

    pred_final = np.argmax(prob_final)
    real = np.argmax(row['soft_label'])

    test_preds_fusion.append(pred_final)
    test_reales_fusion.append(real)

    # --- Guardo toda la información para la tabla ---
    lista_resultados_tabla.append({
        'ID_Meme': id_meme,
        'Soft_VL': np.round(prob_vl, 3).tolist(),
        'Soft_EEG': np.round(prob_eeg_media, 3).tolist(),
        'Soft_ET': np.round(prob_et_media, 3).tolist(),
        'Soft_HR': np.round(prob_hr_media, 3).tolist(),
        'Soft_FINAL': np.round(prob_final, 3).tolist(),
        'Decisión_IA': 'SÍ Sexista' if pred_final == 1 else 'NO Sexista',
        'Realidad': 'SÍ Sexista' if real == 1 else 'NO Sexista'
    })

# RESULTADOS Y MÉTRICAS FINALES
nombres_clases = ['NO Sexista (0)', 'SÍ Sexista (1)']

print("\n" + "*"*35)
print(f" INFORME DEFINITIVO (VL {W_VL*100}% | EEG {W_EEG*100}% | ET {W_ET*100}% | HR {W_HR*100}%)")
print("*"*35)
print(classification_report(test_reales_fusion, test_preds_fusion, target_names=nombres_clases, digits=4))

#  MOSTRAR LA TABLA DE DECISIONES 
df_resultados = pd.DataFrame(lista_resultados_tabla)
print("\nTABLA DE TRANSPARENCIA MULTIMODAL:")
display(df_resultados.head(15))

# MATRIZ DE CONFUSIÓN
cm_fusion = confusion_matrix(test_reales_fusion, test_preds_fusion)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Greens',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases,
            annot_kws={"size": 14})

plt.title(f'Matriz Final: 4 Torres\n(VL+EEG+ET+HR)', fontsize=14, pad=15)
plt.xlabel('Predicción de la IA (Fusión Total)', fontsize=12, labelpad=10)
plt.ylabel('Realidad (Anotadores)', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()